# Offline Reentry Risk / Exposure Prototype

This notebook walks through the reduced academic demo pipeline. It is presentation-oriented and intentionally avoids claiming operational forecast precision.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.cords_loader import load_cords_reentries, select_presentation_cases
from src.spacetrack_client import collect_gp_history
from src.time_window_model import run_time_window_model
from src.corridor import load_path_points, build_corridor_from_points
from src.exposure import run_exposure_analysis


In [ ]:
config = load_config()
reentries = load_cords_reentries(config)
selected_cases = select_presentation_cases(reentries, config)
selected_cases[['case_id', 'object_name', 'norad_id', 'reentry_time_utc']].head()

In [ ]:
gp_histories = collect_gp_history(selected_cases, config)
model_result = run_time_window_model(gp_histories, selected_cases, config)
model_result.metrics if model_result is not None else 'No GP history was available.'

In [ ]:
path_points = load_path_points(config.manual_path_file)
corridor_gdf, path_points_gdf = build_corridor_from_points(path_points, width_km=config.corridor_width_km)
summary_df, country_overlap, land, countries = run_exposure_analysis(corridor_gdf, config)
summary_df

Use `python scripts/run_demo.py` for the full scripted output flow, including saved figures and maps under `outputs/`.